# Tiny Aya L2-Thinker — regional vocabulary trimming and KV-cache optimisation

**Owner:** Agastya Sharma (`ayastaga`) · roadmap items **F** (trim + quantize L2-Thinker) and **C** (KV cache) of the optimisation handoff · reuses `vocabtrim.py` v2 and the same-GPU protocol.

**Why this is not free for a reasoning model.** The 262k tied embedding is read twice per generated token; reasoning traces are 10–30× longer than chat answers, so the per-token saving compounds into minutes on a phone. But the v1 failure on Fire was caused by *formatting tokens that chat output uses and web text never does* (`'।\n\n'`, `')**:'`). Reasoning traces have their own: `\boxed{}`, `**`, LaTeX, numbered steps, in-script punctuation, plus the four think delimiters. So the selection corpus here includes the released translated reasoning data (`CohereLabs/tiny-aya-l2-thinker-multilingual-reasoning`, train split) for the region, and the held-out check includes its test split. Eval prompts (MGSM, Aya skip-200k slice, Macaron) are never in the corpus.

**Pre-registered thresholds (v2 rule, same as Fire).** q4_k_m file ≥ 10% smaller; CPU decode ≥ 5% faster (tg100); held-out fertility increase < 1% per in-region language, including on reasoning traces; f16 greedy identity ≥ 98% on MGSM bn + Aya bn/en vs a same-session control; truncation, doomloop score, L2 rate and MGSM accuracy within ±1 pt / ±0.01 of the control (McNemar n.s.); divergences attributable to a removed token ≈ all of them. Any failure is a result.

**Region.** `REGION` below defaults to Fire's South Asia set (bn hi ta te ur pa are in L2-Thinker's 45 reasoning languages; mr gu ne only in the non-reasoning data). Change the two maps for Earth/Water.

**Prerequisites.** `l2thinker-f16.gguf` from Part 2 of `l2thinker_quant.ipynb` (or run Part 2 here). Same GPU type as the quant notebook if you want to reuse its rows — otherwise everything here is paired within this session and needs nothing from it.

## 0 · Toolchain

Same pins as the safety study (llama.cpp `5f436ddd`, llama-cpp-python 0.3.35). Fails cheaply on missing prerequisites. Also builds the CPU `llama-bench`/`llama-quantize` (phone proxy).

In [1]:
REPO_URL = "https://github.com/ayastaga/tinyaya-quant-safety"
REPO_REF = "main"            # pin to a tag for a citable run; must contain tinyaya_eval/l2.py (and vocabtrim.py)
REPO_DIR = "/content/tinyaya-quant-safety"

import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert r.returncode == 0 and r.stdout.strip(), "GPU runtime required"
print(r.stdout.strip())

NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import hf_hub_download
# L2-Thinker is gated separately from tiny-aya-global: accept its licence on the model page first (CC-BY-NC, research only).
hf_hub_download("CohereLabs/tiny-aya-l2-thinker", "config.json", token=os.environ["HF_TOKEN"])
print("HF: gated access to tiny-aya-l2-thinker confirmed")

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

HF: gated access to tiny-aya-l2-thinker confirmed


In [3]:
import os, subprocess, torch
os.path.exists(REPO_DIR) or subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
!cd {REPO_DIR} && git fetch -q origin && git checkout -q {REPO_REF} && echo "repo @ $(git rev-parse --short HEAD)"
assert os.path.exists(f"{REPO_DIR}/tinyaya_eval/l2.py"), "tinyaya_eval/l2.py missing: copy it into the repo and push (or drop it in now)"

import re
LLAMA_CPP_PIN = re.search(r'LLAMA_CPP_PIN\s*=\s*"(\w+)"', open(f"{REPO_DIR}/tinyaya_eval/config.py").read()).group(1)
os.path.exists("/content/llama.cpp/.git") or subprocess.run(["git", "clone", "-q", "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)
!cd /content/llama.cpp && git fetch -q origin && git checkout -q {LLAMA_CPP_PIN} && echo "llama.cpp @ $(git rev-parse --short HEAD)"
!cd /content/llama.cpp && ( [ -f requirements/requirements-convert_hf_to_gguf.txt ] && pip install -q -r requirements/requirements-convert_hf_to_gguf.txt || pip install -q -r requirements.txt )
!pip install -q -e /content/llama.cpp/gguf-py

cuda = (torch.version.cuda or "12.4").split(".")
cu = f"cu{cuda[0]}{cuda[1]}" if f"cu{cuda[0]}{cuda[1]}" in ("cu121", "cu122", "cu123", "cu124") else "cu124"
!pip install -q llama-cpp-python==0.3.35 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{cu}
!pip install -q -e {REPO_DIR}
!pip install -q -U "protobuf>=5.29,<6" fasttext-wheel datasets pandas
!python -m tinyaya_eval.l2 patch --repo {REPO_DIR}
print("install complete")

repo @ 8350279
llama.cpp @ 5f436dddb
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 149.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires protobuf!=5.28.0,!=5.29.0,<8,>=5, but you have protobuf 4.25.9 which is incompatible.
gradio 6.26.0 requires huggingface-hub<

In [4]:
import llama_cpp
assert llama_cpp.__version__ == "0.3.35", llama_cpp.__version__
if not llama_cpp.llama_supports_gpu_offload():
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --force-reinstall --no-cache-dir llama-cpp-python==0.3.35
    raise SystemExit("CUDA wheel rebuilt from source; restart the session and re-run this cell")
print("GPU offload: OK")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 40441 MiB):
  Device 0: NVIDIA A100-SXM4-40GB, compute capability 8.0, VMM: yes, VRAM: 40441 MiB


GPU offload: OK


> **Restart the session** (Runtime → Restart session) before Part 1. Part 0 is not re-run afterwards.

## 1 · Session state

`MODEL = "l2thinker"`. Generation budget is per model here: `TINYAYA_MAX_NEW=8192`, `TINYAYA_N_CTX=10240` (both are folded into every run identity). 32K ctx would cost ≈2.4 GB of KV per stream at f16; 10240 is what these prompts need.

In [3]:
MODEL      = "l2thinker"
REPO_DIR   = "/content/tinyaya-quant-safety"           # re-declared: the restart after Part 0 clears Part 0's variables
DATA_ROOT  = "/content/tinyaya-eval"
DRIVE_DIR  = "/content/drive/MyDrive/tinyaya-eval"      # "" disables the mirror
MAX_NEW, N_CTX = 8192, 10240

import os
from google.colab import userdata
os.environ.update({"TINYAYA_MODEL": MODEL, "TINYAYA_ROOT": DATA_ROOT, "TINYAYA_DRIVE_DIR": DRIVE_DIR,
                   "TINYAYA_MAX_NEW": str(MAX_NEW), "TINYAYA_N_CTX": str(N_CTX), "HF_TOKEN": userdata.get("HF_TOKEN")})
os.makedirs(f"{DATA_ROOT}/logs", exist_ok=True)
import importlib, tinyaya_eval.config as config, tinyaya_eval.common as common, tinyaya_eval.l2 as l2
importlib.reload(config); importlib.reload(common); importlib.reload(l2)
print(f"model={config.MODEL_NAME} repo={config.HF_REPO} max_new={config.MAX_NEW_TOKENS} n_ctx={config.N_CTX} root={config.ROOT}")
GPU = l2.gpu_name(); print("GPU:", GPU)

model=l2thinker repo=CohereLabs/tiny-aya-l2-thinker max_new=8192 n_ctx=10240 root=/content/tinyaya-eval
GPU: NVIDIA A100-SXM4-40GB


In [4]:
if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    print(f"restored {common.restore_runs()} file(s) from the mirror")
    import shutil, pathlib   # runs_l2 is mirrored file-by-file by l2.gen; restore it too
    src = pathlib.Path(DRIVE_DIR) / "runs_l2"
    if src.exists():
        shutil.copytree(src, pathlib.Path(DATA_ROOT) / "runs_l2", dirs_exist_ok=True); print("restored runs_l2/")

Mounted at /content/drive
restored 49 file(s) from the mirror
restored runs_l2/


In [5]:
!mkdir -p /content/tinyaya-eval/models/gguf && cp -v /content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker-*.gguf /content/tinyaya-eval/models/gguf/
!cp -r /content/drive/MyDrive/tinyaya-eval/calib /content/tinyaya-eval/

'/content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker-f16.gguf' -> '/content/tinyaya-eval/models/gguf/l2thinker-f16.gguf'
'/content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker-q4_0.gguf' -> '/content/tinyaya-eval/models/gguf/l2thinker-q4_0.gguf'
'/content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker-q4_k_m.gguf' -> '/content/tinyaya-eval/models/gguf/l2thinker-q4_k_m.gguf'
'/content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker-q8_0.gguf' -> '/content/tinyaya-eval/models/gguf/l2thinker-q8_0.gguf'


In [6]:
# 1.x · CUDA fix (Colab ships CUDA 13; the llama-cpp-python wheel wants CUDA 12). Run every session, after Part 1.
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
import os, glob, site, subprocess, sys
libs = [d for sp in site.getsitepackages() for d in glob.glob(f"{sp}/nvidia/*/lib")]
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
r = subprocess.run([sys.executable, "-c", "import llama_cpp; print('llama_cpp loads OK')"], capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])

llama_cpp loads OK



In [7]:
# fasttext for Python 3.13: try the wheel, else build 0.9.3 from source (~2 min)
import subprocess, sys, importlib
try:
    import fasttext
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fasttext-wheel"], capture_output=True, text=True)
    importlib.invalidate_caches()
    try:
        import fasttext
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pybind11", "setuptools", "wheel"], check=True)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "fasttext==0.9.3"], capture_output=True, text=True)
        if r.returncode != 0: print(r.stderr[-2000:])
        importlib.invalidate_caches()
        import fasttext
print("fasttext OK:", fasttext.__file__)

fasttext OK: /usr/local/lib/python3.13/dist-packages/fasttext/__init__.py


In [8]:
# update l2.py from the repo, check it's the new version, reload
!cd {REPO_DIR} && git pull -q
import os, importlib
from pathlib import Path
s = (Path(REPO_DIR) / "tinyaya_eval" / "l2.py").read_text()
assert "prompt_seed" in s and "_LID.f.predict" in s, "old l2.py: push the updated file to GitHub first (or copy it in by hand)"
os.environ["TINYAYA_DECODING"] = "greedy"          # this notebook compares trimmed vs untrimmed: greedy
importlib.reload(l2)
print("decoding:", l2.DECODING, "|", l2.lid("বাংলাদেশের রাজধানী ঢাকা এবং এটি একটি বড় শহর"))   # greedy | bn

decoding: greedy | bn


## 2 · Artifacts

HF safetensors → f16 GGUF → q8_0 / q4_k_m / q4_0 with the pinned llama.cpp. Same Cohere2 architecture as Tiny Aya (36 layers, GQA 16/4, tied 262k embedding), so the converter at this pin should handle it; if `convert` fails, the pin must move and **all four precisions move together** (then not comparable to the safety study). Expected sizes ≈ 6.7 / 3.6 / 2.1 / 2.0 GB. The resolved sha printed here is the pin for the write-up (the repo was last modified 2026-09-25).

In [9]:
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=OFF > /dev/null 2>&1 && cmake --build build --target llama-quantize -j$(nproc) 2>&1 | tail -1
# 2 · Convert L2-Thinker to f16 with the three documented fixes (pre-tokenizer alias, shard names, eager load), then quantize
import os, sys, json, shutil, subprocess, inspect, importlib
from pathlib import Path
from gguf import GGUFReader
HF, GGUF = Path("/content/tinyaya-eval/models/hf/l2thinker"), Path("/content/tinyaya-eval/models/gguf")
GGUF.mkdir(parents=True, exist_ok=True); F16 = GGUF / "l2thinker-f16.gguf"

def prep(step):
    r = subprocess.run([sys.executable, "-m", "tinyaya_eval.prepare", step], env=os.environ.copy(), cwd=REPO_DIR)
    assert r.returncode == 0, f"prepare {step} failed; see /content/tinyaya-eval/logs/"
if not (HF / "config.json").exists():
    prep("download")

# fix 1: pre-tokenizer hash alias (tokenizer verified identical to tiny-aya-global in the quant session)
L2_HASH = "9e079518a3ea5a3fe148bd4a570f4962db5db05a20e0cf74632de683e054d5b7"
sys.path.insert(0, "/content/llama.cpp")
base = importlib.import_module("conversion.base")
owner = next(c for _, c in inspect.getmembers(base, inspect.isclass) if "get_vocab_base_pre" in c.__dict__)
f = Path(inspect.getsourcefile(owner)); s = f.read_text()
if L2_HASH not in s:
    fn = s.index("def get_vocab_base_pre"); at = s.index("        res = None\n", fn) + len("        res = None\n")
    s = s[:at] + f'        if chkhsh == "{L2_HASH}":\n            # tiny-aya-l2-thinker: identical to tiny-aya-global except add-EOS on encode\n            res = "tiny_aya"\n' + s[at:]
    f.write_text(s); print("pre-tokenizer alias added")

# fix 2: expose worker-*.safetensors under model-*.safetensors names (symlinks) and rewrite the index
workers = sorted(HF.glob("worker-*.safetensors")); N = len(workers)
rename = {w.name: f"model-{i+1:05d}-of-{N:05d}.safetensors" for i, w in enumerate(workers)}
for old, new in rename.items():
    link = HF / new
    if link.is_symlink() or link.exists(): link.unlink()
    link.symlink_to(HF / old)
idx, orig = HF / "model.safetensors.index.json", HF / "model.safetensors.index.json.orig"
if not orig.exists(): shutil.copy(idx, orig)
j = json.loads(orig.read_text()); j["weight_map"] = {k: rename.get(v, v) for k, v in j["weight_map"].items()}
idx.write_text(json.dumps(j, indent=2)); print(f"{N} shards symlinked, {len(j['weight_map'])} tensors in index")

# fix 3: eager conversion (lazy mode crashes on the all-zero LayerNorm bias check)
F16.unlink(missing_ok=True)
r = subprocess.run([sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", str(HF), "--outfile", str(F16), "--outtype", "f16", "--no-lazy"],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:]); assert r.returncode == 0, "convert failed"
nt = len(GGUFReader(str(F16)).tensors); print(f"f16: {F16.stat().st_size/1e9:.2f} GB, {nt} tensors")
assert nt > 0 and F16.stat().st_size > 6e9
prep("quantize")
prep("inventory")
for p in ("q8_0", "q4_k_m", "q4_0"):
    q = GGUF / f"l2thinker-{p}.gguf"
    print(f"{p}: {q.stat().st_size/1e9:.2f} GB, {len(GGUFReader(str(q)).tensors)} tensors")

r = subprocess.run([sys.executable, "-c", "from llama_cpp import Llama; import sys; Llama(model_path=sys.argv[1], n_ctx=512, n_gpu_layers=-1, verbose=False); print('LOAD OK')", str(F16)],
                   capture_output=True, text=True, env=os.environ.copy())
print(r.stdout.strip() or r.stderr[-800:])
print("pinned revision:", common.revision())

[100%] Built target llama-quantize
pre-tokenizer alias added
4 shards symlinked, 471 tensors in index
70%|██████▉   | 4.68G/6.70G [00:10<00:03, 567Mbyte/s]
Writing:  72%|███████▏  | 4.82G/6.70G [00:10<00:03, 563Mbyte/s]
Writing:  74%|███████▍  | 4.95G/6.70G [00:10<00:02, 582Mbyte/s]
Writing:  76%|███████▌  | 5.09G/6.70G [00:10<00:02, 659Mbyte/s]
Writing:  78%|███████▊  | 5.22G/6.70G [00:10<00:01, 758Mbyte/s]
Writing:  80%|███████▉  | 5.36G/6.70G [00:11<00:01, 782Mbyte/s]
Writing:  82%|████████▏ | 5.49G/6.70G [00:11<00:01, 648Mbyte/s]
Writing:  83%|████████▎ | 5.58G/6.70G [00:11<00:01, 568Mbyte/s]
Writing:  85%|████████▍ | 5.67G/6.70G [00:11<00:01, 515Mbyte/s]
Writing:  86%|████████▌ | 5.76G/6.70G [00:12<00:01, 482Mbyte/s]
Writing:  87%|████████▋ | 5.85G/6.70G [00:12<00:01, 449Mbyte/s]
Writing:  89%|████████▊ | 5.94G/6.70G [00:12<00:01, 431Mbyte/s]
Writing:  89%|████████▉ | 5.99G/6.70G [00:12<00:01, 420Mbyte/s]
Writing:  90%|█████████ | 6.04G/6.70G [00:12<00:01, 411Mbyte/s]
Writing:  91

In [10]:
PRECISIONS = ["f16", "q4_k_m"]

## 3 · Preflight gate (thinking mode)

For every GGUF: the template carries the trigger and ends with `<|START_THINKING|>`; the four think delimiters are single tokens; the stop set has `<|END_RESPONSE|>` and **not** `<|END_THINKING|>`; a probe closes its think block, produces an answer, stops, and is deterministic. Nothing runs past a failure (a missed stop id would send every trace to the cap and make truncation meaningless — the trimming study's fresh-prompt landmine).

In [11]:
import subprocess, sys
fails = []
for prec in PRECISIONS:
    r = subprocess.run([sys.executable, "-m", "tinyaya_eval.l2", "probe", "--precision", prec], capture_output=True, text=True, env=os.environ.copy(), cwd=REPO_DIR)
    print(r.stdout, end="")
    if r.returncode != 0:
        fails.append(prec); print("  stderr:", r.stderr[-1500:])
assert not fails, f"preflight failed: {fails}"

--- f16 --- trigger in template: True | ends with START_THINKING: True
think ids: {'start': 261006, 'end': 261007, 'resp_start': 261000, 'resp_end': 261001} | stop ids: [(3, '<EOS_TOKEN>'), (6, '<|END_OF_TURN_TOKEN|>'), (261001, '<|END_RESPONSE|>')]
decoding=greedy
finish=stop stop=<|END_RESPONSE|> think_closed=True n_trace=22 n_answer=9
trace tail: 'ব্যবহারকারী একটি সাধারণ গণিতের প্রশ্ন করেছেন: 2+2 কত? উত্তর হল 4।'
answer   : '2 + 2 = 4।'
deterministic: True => PASS
--- q4_k_m --- trigger in template: True | ends with START_THINKING: True
think ids: {'start': 261006, 'end': 261007, 'resp_start': 261000, 'resp_end': 261001} | stop ids: [(3, '<EOS_TOKEN>'), (6, '<|END_OF_TURN_TOKEN|>'), (261001, '<|END_RESPONSE|>')]
decoding=greedy
finish=stop stop=<|END_RESPONSE|> think_closed=True n_trace=28 n_answer=9
trace tail: 'ব্যবহারকারী একটি সরল গণিতের প্রশ্ন জিজ্ঞাসা করেছেন: 2+2 = 4। উত্তর প্রদান করুন।'
answer   : '2 + 2 = 4।'
deterministic: True => PASS


In [12]:
# Runner used by every generation cell: one process per job, at most MAX_PROCS at once, resumable, logs under logs/.
import subprocess, sys, time
from pathlib import Path
LOGS = Path(DATA_ROOT) / "logs"
def run_jobs(jobs, max_procs, env=None, tag="pilot"):
    """jobs: list of dicts(precision, dataset, langs, n, variant='L2', prefix='', model=None)."""
    queue, live, rc = list(jobs), {}, {}
    while queue or live:
        while queue and len(live) < max_procs:
            j = queue.pop(0)
            e = {**os.environ, **(env or {})}
            if j.get("model"):
                e["TINYAYA_MODEL"] = j["model"]
            name = f"{e['TINYAYA_MODEL']}_{j['dataset']}_{j.get('variant','L2')}_{j['precision']}"
            cmd = [sys.executable, "-m", "tinyaya_eval.l2", "gen", "--precision", j["precision"], "--dataset", j["dataset"],
                   "--langs", ",".join(j["langs"]), "--n", str(j["n"]), "--variant", j.get("variant", "L2"), "--tag", tag]
            if j.get("prefix"):
                cmd += ["--prefix", j["prefix"]]
            live[name] = subprocess.Popen(cmd, stdout=open(LOGS / f"gen_{name}.log", "a"), stderr=subprocess.STDOUT, env=e, cwd=REPO_DIR)
        time.sleep(60)
        for name, p in list(live.items()):
            if p.poll() is not None:
                rc[name] = p.returncode; del live[name]
        print(time.strftime("%H:%M"), "live:", list(live), "| done:", {k: v for k, v in rc.items()}, flush=True)
    bad = [k for k, v in rc.items() if v != 0]
    assert not bad, f"failed: {bad} (see logs/gen_<name>.log)"
    return rc
print("run_jobs ready")

run_jobs ready


In [13]:
import json, statistics
from pathlib import Path
import pandas as pd
def rows_df(tag, model=None):
    d = Path(DATA_ROOT) / "runs_l2" / (model or config.MODEL_NAME) / tag
    recs = [json.loads(l) for p in sorted(d.glob("*.jsonl")) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
    df = pd.DataFrame(recs)
    if len(df):
        df["tier"] = df.lang.map(l2.LANG_TIER)
    return df
def add_lid(df):
    if "trace_lang" not in df:
        df["trace_lang"] = [l2.lid(t) if isinstance(t, str) and t.strip() else None for t in df.trace]
        df["l2"] = df.trace_lang == df.lang
    return df
def same_gpu(df):
    g = df.gpu.unique(); assert len(g) == 1, f"mixed GPUs {g}: every paired comparison must come from one GPU type/session"; return g[0]
print("loaders ready")

loaders ready


## 10 · Trim setup: names, paths, tools

In [14]:
# 2.x · CPU build of llama-quantize and llama-bench at the pinned commit, copied to build/bin
import subprocess
from pathlib import Path
LLAMA = Path("/content/llama.cpp")
r = subprocess.run(f"cd {LLAMA} && cmake -B build-cpu -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_SERVER=OFF "
                   f"&& cmake --build build-cpu --target llama-quantize llama-bench -j$(nproc)", shell=True, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:])
assert r.returncode == 0, "build failed (see output above)"
(LLAMA / "build" / "bin").mkdir(parents=True, exist_ok=True)
for tool in ("llama-quantize", "llama-bench"):
    src, dst = LLAMA / "build-cpu" / "bin" / tool, LLAMA / "build" / "bin" / tool
    subprocess.run(["cp", "-f", str(src), str(dst)], check=True)
    print(tool, "->", dst, dst.is_file())
print("pin:", subprocess.run(["git", "-C", str(LLAMA), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

on.dir/jinja/lexer.cpp.o
[ 95%] Building CXX object common/CMakeFiles/llama-common.dir/jinja/parser.cpp.o
[ 95%] Building CXX object common/CMakeFiles/llama-common.dir/jinja/runtime.cpp.o
[ 95%] Building CXX object common/CMakeFiles/llama-common.dir/jinja/value.cpp.o
[ 97%] Building CXX object common/CMakeFiles/llama-common.dir/jinja/string.cpp.o
[ 97%] Building CXX object common/CMakeFiles/llama-common.dir/jinja/caps.cpp.o
[ 97%] Linking CXX shared library ../bin/libllama-common.so
[ 97%] Built target llama-common
[ 97%] Building CXX object tools/quantize/CMakeFiles/llama-quantize-impl.dir/quantize.cpp.o
[ 97%] Linking CXX shared library ../../bin/libllama-quantize-impl.so
[ 97%] Built target llama-quantize-impl
[ 97%] Building CXX object tools/quantize/CMakeFiles/llama-quantize.dir/main.cpp.o
[100%] Linking CXX executable ../../bin/llama-quantize
[100%] Built target llama-quantize
[  2%] Built target llama-common-base
[  2%] Built target cpp-httplib
[ 10%] Built target ggml-base
[ 23

In [15]:
import os, sys, json, hashlib, subprocess, importlib, shutil
from pathlib import Path
REPO  = Path(REPO_DIR); ROOT = Path(DATA_ROOT); LLAMA = Path("/content/llama.cpp")
BASE, TRIM, CTRL = "l2thinker", "l2thinker_trim", "l2thinker_ctrl"      # registered by `l2 patch`
GGUF, TRIMDIR, LOGS = ROOT / "models" / "gguf", ROOT / "calib" / "trim" / BASE, ROOT / "logs"
for d in (TRIMDIR / "corpus", TRIMDIR / "heldout", LOGS): d.mkdir(parents=True, exist_ok=True)
assert (REPO / "tinyaya_eval" / "vocabtrim.py").exists(), "copy vocabtrim.py into tinyaya_eval/ and push it"
F16 = GGUF / f"{BASE}-f16.gguf"; assert F16.exists(), f"{F16} missing: run Part 2"
QUANTIZE_BIN, BENCH_BIN = LLAMA / "build" / "bin" / "llama-quantize", LLAMA / "build" / "bin" / "llama-bench"
assert QUANTIZE_BIN.is_file() and BENCH_BIN.is_file()
assert subprocess.run(["git", "-C", str(LLAMA), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() == config.LLAMA_CPP_PIN
# sanity for the "preserved by construction" argument (handoff §5): tied embedding, n_embd % 256 == 0, byte-level BPE
from llama_cpp import Llama
vt = Llama(model_path=str(F16), vocab_only=True, verbose=False)
md = vt.metadata; print({k: md.get(k) for k in ("tokenizer.ggml.model", "cohere2.embedding_length", "general.architecture")})
assert md.get("tokenizer.ggml.model") == "gpt2", "vocabtrim's ancestor closure assumes byte-level BPE with merges"
del vt
print("base:", BASE, "| trim:", TRIM, "| f16:", f"{F16.stat().st_size/1e9:.2f} GB")

llama_context: n_ctx_seq (512) > n_ctx_train (0) -- possible training context overflow


{'tokenizer.ggml.model': 'gpt2', 'cohere2.embedding_length': '2048', 'general.architecture': 'cohere2'}
base: l2thinker | trim: l2thinker_trim | f16: 6.71 GB


## 11 · Selection corpus

FineWeb-2 + Aya Collection per region language + FineWeb-Edu English + code (as in the Fire trim), **plus** the region's translated reasoning traces (question + thinking + output) from the released MR data, train split. Held-out: FineWeb-2 test, English, and the MR **test** split (reasoning traces the selection never saw). MultiJail, MGSM, Macaron and the Aya skip-200k eval slice are excluded.

In [ ]:
from datasets import load_dataset, get_dataset_config_names
REGION = {  # Fire = South Asia (Tiny Aya paper Table 1). Earth/Water: change these maps.
    "bn": "ben_Beng", "hi": "hin_Deva", "mr": "mar_Deva", "gu": "guj_Gujr", "pa": "pan_Guru",
    "ta": "tam_Taml", "te": "tel_Telu", "ne": "npi_Deva", "ur": "urd_Arab"}
SCRIPTS = "BENGALI,DEVANAGARI,GUJARATI,GURMUKHI,TAMIL,TELUGU,ARABIC"     # every token entirely in these scripts is kept (v2 rule)
MR_LANGS = [l for l in REGION if l in l2.L2_45]                          # reasoning data exists only for the 45 L2 languages
MB_WEB, MB_AYA, MB_EN, MB_CODE, MB_HELD, MB_MR = 40, 10, 80, 40, 3, 20

def stream_to(path, ds, field, mb):
    if path.exists() and path.stat().st_size >= mb * 1e6 * 0.9: return path.stat().st_size
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for row in ds:
            t = (row.get(field) or "").strip() if isinstance(row, dict) else ""
            if t: f.write(t + "\n"); n += len(t.encode())
            if n >= mb * 1e6: break
    return n

aya_avail = set(get_dataset_config_names("CohereForAI/aya_collection_language_split"))
for lang, cfg_ in REGION.items():
    n = stream_to(TRIMDIR / "corpus" / f"web_{lang}.txt", load_dataset("HuggingFaceFW/fineweb-2", name=cfg_, split="train", streaming=True), "text", MB_WEB)
    h = stream_to(TRIMDIR / "heldout" / f"web_{lang}.txt", load_dataset("HuggingFaceFW/fineweb-2", name=cfg_, split="test", streaming=True), "text", MB_HELD)
    a = 0
    if l2.AYA_NAMES.get(lang) in aya_avail:
        ds = load_dataset("CohereForAI/aya_collection_language_split", l2.AYA_NAMES[lang], split="train", streaming=True)
        a = stream_to(TRIMDIR / "corpus" / f"aya_{lang}.txt", ({"text": f"{r.get('inputs','')}\n{r.get('targets','')}"} for r in ds), "text", MB_AYA)
    print(f"{lang}: web {n/1e6:.0f} MB, aya {a/1e6:.0f} MB, heldout {h/1e6:.1f} MB")
# reasoning traces: the distribution a trimmed L2-Thinker must reproduce. Train split -> corpus; test split -> held-out.
for lang in MR_LANGS:
    for split, dst, mb in (("train", TRIMDIR / "corpus" / f"mr_{lang}.txt", MB_MR), ("test", TRIMDIR / "heldout" / f"mr_{lang}.txt", MB_HELD)):
        try:
            ds = load_dataset("CohereLabs/tiny-aya-l2-thinker-multilingual-reasoning", data_files=f"data/{lang}/{split}.parquet", split="train", streaming=True)
            n = stream_to(dst, ({"text": f"{r['question']}\n<|START_THINKING|>{r['thinking']}<|END_THINKING|><|START_RESPONSE|>{r['output']}<|END_RESPONSE|>"} for r in ds), "text", mb)
            print(f"mr {lang} {split}: {n/1e6:.1f} MB")
        except Exception as e:
            print(f"mr {lang} {split}: skipped ({str(e)[:80]})")
print("en:", stream_to(TRIMDIR / "corpus" / "web_en.txt", load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True), "text", MB_EN) / 1e6, "MB")
print("code:", stream_to(TRIMDIR / "corpus" / "code.txt", load_dataset("codeparrot/codeparrot-clean-valid", split="train", streaming=True), "content", MB_CODE) / 1e6, "MB")
stream_to(TRIMDIR / "heldout" / "web_en.txt", load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True).skip(20000), "text", MB_HELD)
# held-out: the harness's own prompt material (never in the corpus)
(TRIMDIR / "heldout" / "mgsm_bn.txt").write_text("\n".join(it["prompt"] for it in l2.load_items("mgsm", ["bn"], 250)), encoding="utf-8")
(TRIMDIR / "heldout" / "mgsm_en.txt").write_text("\n".join(it["prompt"] for it in l2.load_items("mgsm", ["en"], 250)), encoding="utf-8")
print("corpus MB:", sum(p.stat().st_size for p in (TRIMDIR / "corpus").glob("*.txt")) / 1e6)

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

bn: web 40 MB, aya 10 MB, heldout 3.0 MB
hi: web 40 MB, aya 10 MB, heldout 3.0 MB
mr: web 40 MB, aya 10 MB, heldout 3.0 MB
gu: web 40 MB, aya 10 MB, heldout 3.0 MB
pa: web 40 MB, aya 0 MB, heldout 3.0 MB
ta: web 40 MB, aya 10 MB, heldout 3.0 MB
te: web 40 MB, aya 10 MB, heldout 3.0 MB
ne: web 40 MB, aya 10 MB, heldout 3.0 MB
ur: web 40 MB, aya 10 MB, heldout 3.0 MB


README.md: 0.00B [00:00, ?B/s]

mr bn train: 20.0 MB
mr bn test: 3.0 MB
mr hi train: 20.0 MB
mr hi test: 3.0 MB
mr pa train: 20.0 MB
mr pa test: 3.0 MB
mr ta train: 20.0 MB
mr ta test: 3.0 MB
mr te train: 20.0 MB
mr te test: 3.0 MB
mr ur train: 20.0 MB
mr ur test: 2.2 MB


...

en/train-00000-of-00001.parquet:   0%|          | 0.00/4.97k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

corpus MB: 680.195275


## 12 · Select, apply, pin

v2 rule: corpus tokens + special/control + 256 byte tokens + every no-letter token + every token entirely in the region's scripts + BPE-ancestor closure. `apply` aborts if any `*_token_id` (incl. the think delimiters) would be removed. The revision string is base sha + keep hash, so run identities never collide with the base model's.

In [16]:
KEEP = TRIMDIR / f"{TRIM}_keep.json"; OUT16 = GGUF / f"{TRIM}-f16.gguf"
if not KEEP.exists():
    subprocess.run([sys.executable, "-m", "tinyaya_eval.vocabtrim", "select", "--src", str(F16), "--corpus", str(TRIMDIR / "corpus" / "*.txt"),
                    "--out", str(KEEP), "--keep-no-letters", "--keep-scripts", SCRIPTS], check=True, cwd=str(REPO))
stats = json.loads(KEEP.read_text())["stats"]; print({k: stats[k] for k in ("vocab", "used_in_corpus", "kept", "kept_pct", "keep_sha256")})
keep = set(json.loads(KEEP.read_text())["keep"])
vt = Llama(model_path=str(F16), vocab_only=True, verbose=False)
for s in l2.THINK.values():
    tid = vt.tokenize(s.encode(), add_bos=False, special=True)[0]; assert tid in keep, f"{s} ({tid}) not in keep set"
del vt; print("think delimiters kept")
if not OUT16.exists():
    subprocess.run([sys.executable, "-m", "tinyaya_eval.vocabtrim", "apply", "--src", str(F16), "--keep", str(KEEP), "--dst", str(OUT16)], check=True, cwd=str(REPO))
print(json.loads(Path(str(OUT16) + ".trim.json").read_text()))
base_rev = common.revision_file(BASE).read_text().strip()
rf = common.revision_file(TRIM); rf.write_text(f"{base_rev}+trim{stats['keep_sha256']}"); common.mirror(rf)
for p in (KEEP, Path(str(OUT16) + ".trim.json")): common.mirror(p)
print("revision:", rf.read_text())

{'vocab': 262144, 'used_in_corpus': 125014, 'kept': 138105, 'kept_pct': 52.68, 'keep_sha256': 'bfadfaf070959033'}


llama_context: n_ctx_seq (512) > n_ctx_train (0) -- possible training context overflow


think delimiters kept
{'src': '/content/tinyaya-eval/models/gguf/l2thinker-f16.gguf', 'keep_file': '/content/tinyaya-eval/calib/trim/l2thinker/l2thinker_trim_keep.json', 'keep_sha256': 'bfadfaf070959033', 'vocab_before': 262144, 'vocab_after': 138105, 'merges_before': 260729, 'merges_after': 136690, 'bytes_before': 6710484480, 'bytes_after': 6196508672}
revision: e9feb287c4a4755ac5ee4d085b8788a78811604f+trimbfadfaf070959033


## 13 · Tokenization drift on held-out text (web, English, reasoning traces, MGSM prompts)

In [17]:
rows = []
for p in sorted((TRIMDIR / "heldout").glob("*.txt")):
    r = subprocess.run([sys.executable, "-m", "tinyaya_eval.vocabtrim", "verify", "--src", str(F16), "--dst", str(OUT16), "--texts", str(p)], capture_output=True, text=True, cwd=str(REPO))
    j = json.loads(r.stdout[r.stdout.rfind("{"):]) if "{" in r.stdout else {}
    rows.append((p.stem, j.get("lines"), j.get("lines_tokenized_identically_pct"), j.get("fertility_increase_pct")))
print(f"{'heldout':16s} {'lines':>7s} {'identical %':>12s} {'fertility +%':>13s}")
for name, n, ident, fert in rows:
    print(f"{name:16s} {n or 0:7d} {ident or 0:12.2f} {fert or 0:13.2f}{'  <-- over 1%' if (fert or 0) > 1.0 else ''}")
print("\nrule: every in-region language, INCLUDING mr_<lang> reasoning traces, under 1% fertility increase")

heldout            lines  identical %  fertility +%
mgsm_bn              250       100.00          0.00
mgsm_en              250        99.20          0.02
mr_bn              12256        99.84          0.01
mr_hi              12666        99.92          0.00
mr_pa              11952        99.86          0.00
mr_ta              10933        99.91          0.00
mr_te              12712        99.77          0.01
mr_ur              11660        99.85          0.00
web_bn              5112        99.98          0.00
web_en             13530        99.99          0.00
web_gu              5401        99.83          0.00
web_hi              5517        99.40          0.01
web_mr              4742        99.87          0.00
web_ne              4809        99.94          0.00
web_pa              5507        99.73          0.00
web_ta              5344        99.87          0.00
web_te              4776        99.92          0.00
web_ur              7300        99.86          0.03

rule: every

## 14 · Quantize the trimmed model, sizes, CPU decode speed (phone proxy)

In [18]:
env_trim = {**os.environ, "TINYAYA_MODEL": TRIM, "TINYAYA_CALIB": ""}
subprocess.run([sys.executable, "-m", "tinyaya_eval.prepare", "quantize"], check=True, env=env_trim, cwd=str(REPO))
print(f"{'file':32s} {'GB':>6s} {'vs base':>8s}")
for prec in ("f16", "q8_0", "q4_k_m", "q4_0"):
    a, b = GGUF / f"{BASE}-{prec}.gguf", GGUF / f"{TRIM}-{prec}.gguf"
    if a.exists() and b.exists(): print(f"{b.name:32s} {b.stat().st_size/1e9:6.2f} {100*(b.stat().st_size/a.stat().st_size-1):7.1f}%")
def bench(model, ngl, threads, n=100):
    r = subprocess.run([str(BENCH_BIN), "-m", str(model), "-p", "100", "-n", str(n), "-ngl", str(ngl), "-t", str(threads), "-r", "5", "-o", "json"], capture_output=True, text=True)
    out = json.loads(r.stdout[r.stdout.find("["):]) if "[" in r.stdout else []
    return {d.get("test", d.get("n_gen", "?")): round(d["avg_ts"], 1) for d in out} or r.stderr[-400:]
for name in (BASE, TRIM):
    print(f"cpu {name}-q4_k_m:", bench(GGUF / f"{name}-q4_k_m.gguf", 0, os.cpu_count()))
print("\nrule: q4_k_m >= 10% smaller and CPU tg100 >= 5% faster; report tg only (pp on 2 cores is noise); add peak RSS on a real device")

file                                 GB  vs base
l2thinker_trim-f16.gguf            6.20    -7.7%
l2thinker_trim-q8_0.gguf           3.29    -7.7%
l2thinker_trim-q4_k_m.gguf         1.93   -10.0%
l2thinker_trim-q4_0.gguf           1.82   -10.5%
cpu l2thinker-q4_k_m: {0: 64.1, 100: 16.1}
cpu l2thinker_trim-q4_k_m: {0: 76.2, 100: 17.4}

rule: q4_k_m >= 10% smaller and CPU tg100 >= 5% faster; report tg only (pp on 2 cores is noise); add peak RSS on a real device


## 15 · Free generation, same session, same GPU: trimmed vs control

The untrimmed model is regenerated in this session as `l2thinker_ctrl` (symlinked GGUFs, revision `<sha>+ctrl`). MGSM bn (accuracy) + Aya bn/en (L2 rate, doomloop, truncation), thinking mode, greedy, at f16 and q4_k_m. Identity ≥ 98% at f16, every other metric within the control's noise, divergences explained by a removed token.

In [19]:
# Memory-aware runner: start a job only if its VRAM fits next to the live ones. Resumes existing rows.
import os, sys, time, subprocess, importlib
from pathlib import Path
os.environ["TINYAYA_DECODING"] = "greedy"          # trim-vs-control identity test: greedy, not sampling
importlib.reload(l2); print("decoding:", l2.DECODING)
r = subprocess.run(["pgrep", "-fa", "[t]inyaya_eval.l2 gen"], capture_output=True, text=True)   # [t] stops pgrep matching itself
assert not r.stdout.strip(), f"old jobs still running, pkill them first:\n{r.stdout}"

BUDGET_GB = 37
COST = {"f16": 8.0, "q8_0": 5.0, "q4_k_m": 3.5, "q4_0": 3.5}   # weights + KV at n_ctx 10240, measured-ish
LOGS = Path(DATA_ROOT) / "logs"

def run_jobs_budget(jobs, tag):
    queue, live, rc = list(jobs), {}, {}
    while queue or live:
        used = sum(COST[j["precision"]] for _, j in live.values())
        for j in list(queue):
            if used + COST[j["precision"]] <= BUDGET_GB:
                queue.remove(j); used += COST[j["precision"]]
                e = {**os.environ, "TINYAYA_MODEL": j["model"]}
                name = f"{j['model']}_{j['dataset']}_L2_{j['precision']}"
                cmd = [sys.executable, "-m", "tinyaya_eval.l2", "gen", "--precision", j["precision"], "--dataset", j["dataset"],
                       "--langs", ",".join(j["langs"]), "--n", str(j["n"]), "--variant", "L2", "--tag", tag]
                live[name] = (subprocess.Popen(cmd, stdout=open(LOGS / f"gen_{name}.log", "a"), stderr=subprocess.STDOUT, env=e, cwd=REPO_DIR), j)
        for name, (p, _) in list(live.items()):
            if p.poll() is not None:
                rc[name] = p.returncode; del live[name]
        #print(time.strftime("%H:%M"), "live:", list(live), "| queued:", len(queue), "| done:", rc, flush=True)
    bad = [k for k, v in rc.items() if v != 0]
    assert not bad, f"failed: {bad} (see logs/gen_<name>.log)"

N_MGSM, N_AYA = 250, 100                            # Option B sizes; A100: 250, 100
jobs  = [{"model": m, "precision": p, "dataset": "mgsm", "langs": ["bn"], "n": N_MGSM} for m in (TRIM, CTRL) for p in ("f16", "q4_k_m")]
jobs += [{"model": m, "precision": p, "dataset": "aya",  "langs": ["bn", "en"], "n": N_AYA} for m in (TRIM, CTRL) for p in ("f16", "q4_k_m")]
run_jobs_budget(jobs, tag="trim_a100")                  # Option A on an A100: tag="trim_a100"

decoding: greedy


AssertionError: failed: ['l2thinker_ctrl_mgsm_L2_q4_k_m', 'l2thinker_ctrl_mgsm_L2_f16', 'l2thinker_trim_aya_L2_f16', 'l2thinker_trim_aya_L2_q4_k_m', 'l2thinker_ctrl_aya_L2_q4_k_m', 'l2thinker_ctrl_aya_L2_f16'] (see logs/gen_<name>.log)

In [24]:
import subprocess
from pathlib import Path
GGUF = Path(DATA_ROOT) / "models" / "gguf"
rf = common.revision_file(CTRL); rf.write_text(common.revision_file(BASE).read_text().strip() + "+ctrl"); common.mirror(rf)
for prec in ("f16", "q4_k_m"):
    src, link = GGUF / f"{BASE}-{prec}.gguf", GGUF / f"{CTRL}-{prec}.gguf"
    assert src.exists(), f"missing base {src}"
    if link.is_symlink() or link.exists(): link.unlink()
    link.symlink_to(src)
    assert link.exists(), f"dangling link {link}"
    print(link.name, "->", link.resolve().name)

l2thinker_ctrl-f16.gguf -> l2thinker-f16.gguf
l2thinker_ctrl-q4_k_m.gguf -> l2thinker-q4_k_m.gguf


In [25]:
   !cp -v /content/tinyaya-eval/models/gguf/l2thinker_trim-*.gguf /content/tinyaya-eval/models/gguf/*.trim.json /content/drive/MyDrive/tinyaya-eval/models/gguf/

'/content/tinyaya-eval/models/gguf/l2thinker_trim-f16.gguf' -> '/content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker_trim-f16.gguf'
'/content/tinyaya-eval/models/gguf/l2thinker_trim-q4_0.gguf' -> '/content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker_trim-q4_0.gguf'
'/content/tinyaya-eval/models/gguf/l2thinker_trim-q4_k_m.gguf' -> '/content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker_trim-q4_k_m.gguf'
'/content/tinyaya-eval/models/gguf/l2thinker_trim-q8_0.gguf' -> '/content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker_trim-q8_0.gguf'
'/content/tinyaya-eval/models/gguf/l2thinker_trim-f16.gguf.trim.json' -> '/content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker_trim-f16.gguf.trim.json'


In [30]:
import numpy as np, pandas as pd
TAG = "trim_a100"
ct, tr = add_lid(rows_df(TAG, CTRL)), add_lid(rows_df(TAG, TRIM))
for d in (ct, tr):
    m = d.dataset == "mgsm"
    d.loc[m, "correct"] = [l2.mgsm_correct(a, g) for a, g in zip(d.loc[m, "answer"], d.loc[m, "gold"])]

both = pd.concat([ct, tr])
assert both.gpu.nunique() == 1, both.gpu.unique()
assert set(both.decoding) == {"greedy"}, set(both.decoding)
print("gpu:", both.gpu.iloc[0], "| rows ctrl/trim:", len(ct), len(tr))

def pick(*names):
    for n in names:
        if n in ct.columns: return n
    raise KeyError(f"none of {names} in columns: {list(ct.columns)}")
ID, LANG = pick("id", "prompt_id", "qid"), pick("lang", "language")
LID, LOOP = pick("trace_lid", "lid"), pick("doomloop", "doomloop_score", "loop")
TRUNC = pick("truncated", "hit_max", "trunc")
TXT = [c for c in ("trace", "answer", "text", "output") if c in ct.columns]
print("using:", dict(id=ID, lang=LANG, lid=LID, loop=LOOP, trunc=TRUNC, text=TXT))

KEY = ["dataset", "precision", LANG, ID]
assert not ct.duplicated(KEY).any() and not tr.duplicated(KEY).any(), "duplicate rows"
p = ct.merge(tr, on=KEY, suffixes=("_c", "_t"), validate="1:1")
assert len(p) == len(ct) == len(tr), (len(p), len(ct), len(tr))
p["same"] = np.logical_and.reduce([p[f"{c}_c"] == p[f"{c}_t"] for c in TXT])
for s in ("c", "t"):
    p[f"l2_{s}"] = p[f"{LID}_{s}"] == p[LANG]

rows = []
for (ds, prec, lang), g in p.groupby(["dataset", "precision", LANG]):
    r = dict(dataset=ds, prec=prec, lang=lang, n=len(g), identity=100 * g.same.mean(),
             d_trunc=100 * (g[f"{TRUNC}_t"].mean() - g[f"{TRUNC}_c"].mean()),
             d_L2=100 * (g.l2_t.mean() - g.l2_c.mean()),
             d_loop=g[f"{LOOP}_t"].mean() - g[f"{LOOP}_c"].mean())
    if ds == "mgsm":
        r.update(acc_c=100 * g.correct_c.mean(), acc_t=100 * g.correct_t.mean(),
                 d_acc=100 * (g.correct_t.mean() - g.correct_c.mean()),
                 mcnemar=l2.paired_binary(g.correct_c.astype(bool).values, g.correct_t.astype(bool).values))
    rows.append(r)
res = pd.DataFrame(rows)
pd.set_option("display.width", 200); print(res.round(3).to_string(index=False))

print("\nPre-registered checks:")
f16 = res[res.prec == "f16"]
print(f"  f16 greedy identity >= 98%: {f16.identity.round(2).tolist()} ->", "PASS" if (f16.identity >= 98).all() else "FAIL")
for col, tol in (("d_trunc", 1), ("d_L2", 1), ("d_acc", 1), ("d_loop", 0.01)):
    v = res[col].dropna()
    print(f"  {col} within ±{tol}: {v.round(3).tolist()} ->", "PASS" if (v.abs() <= tol).all() else "FAIL")
print("  McNemar n.s.: check the mcnemar column (p > 0.05)")
print(f"\n{(~p.same).sum()} divergent pairs; first few:")
print(p.loc[~p.same, KEY].head(10).to_string(index=False))

gpu: NVIDIA A100-SXM4-40GB | rows ctrl/trim: 900 900


KeyError: "none of ('trace_lid', 'lid') in columns: ['decoding', 'seed', 'id', 'lang', 'gold', 'aux', 'dataset', 'variant', 'prefix', 'precision', 'model', 'revision', 'gpu', 'max_new', 'n_ctx', 'trace', 'answer', 'n_prompt', 'n_trace', 'n_answer', 'n_gen', 'finish', 'stop_token', 'truncated', 'think_closed', 'doomloop', 'correct', 'tier', 'trace_lang', 'l2']"

In [28]:
import os, sys, time, subprocess
from pathlib import Path
r = subprocess.run(["pgrep", "-fa", "[t]inyaya_eval.l2 gen"], capture_output=True, text=True)
assert not r.stdout.strip(), f"old jobs still running, pkill them first:\n{r.stdout}"
assert l2.DECODING == "greedy", l2.DECODING
GGUF = Path(DATA_ROOT) / "models" / "gguf"
for prec in ("f16", "q4_k_m"):
    assert (GGUF / f"{CTRL}-{prec}.gguf").exists(), f"missing ctrl link {prec}: run the control-links cell first"

BUDGET_GB = 36.0                                           # A100 40 GB
COST = {"f16": 8.0, "q8_0": 5.0, "q4_k_m": 3.5, "q4_0": 3.5}
LOGS = Path(DATA_ROOT) / "logs"

def run_jobs_budget(jobs, tag):
    queue, live, rc = list(jobs), {}, {}
    while queue or live:
        used = sum(COST[j["precision"]] for _, j in live.values())
        for j in list(queue):
            if used + COST[j["precision"]] <= BUDGET_GB:
                queue.remove(j); used += COST[j["precision"]]
                m = j.get("model", config.MODEL_NAME); v = j.get("variant", "L2")
                name = f"{m}_{j['dataset']}_{v}_{j['precision']}"
                cmd = [sys.executable, "-m", "tinyaya_eval.l2", "gen", "--precision", j["precision"], "--dataset", j["dataset"],
                       "--langs", ",".join(j["langs"]), "--n", str(j["n"]), "--variant", v, "--tag", tag]
                live[name] = (subprocess.Popen(cmd, stdout=open(LOGS / f"gen_{name}.log", "a"), stderr=subprocess.STDOUT,
                                               env={**os.environ, "TINYAYA_MODEL": m}, cwd=REPO_DIR), j)
                print(time.strftime("%H:%M"), "started", name, flush=True); time.sleep(60)
        time.sleep(30)
        for name, (p, _) in list(live.items()):
            if p.poll() is not None:
                rc[name] = p.returncode; del live[name]
                print(time.strftime("%H:%M"), "finished", name, "rc", rc[name], "| live", len(live), "| queued", len(queue), flush=True)
    bad = [k for k, v in rc.items() if v != 0]
    assert not bad, f"failed: {bad} (see logs/gen_<name>.log)"

N_MGSM, N_AYA = 250, 100
jobs  = [{"model": m, "precision": p, "dataset": "mgsm", "langs": ["bn"], "n": N_MGSM} for m in (TRIM, CTRL) for p in ("f16", "q4_k_m")]
jobs += [{"model": m, "precision": p, "dataset": "aya",  "langs": ["bn", "en"], "n": N_AYA} for m in (TRIM, CTRL) for p in ("f16", "q4_k_m")]
run_jobs_budget(jobs, tag="trim_a100")

01:42 started l2thinker_trim_mgsm_L2_f16
01:43 started l2thinker_trim_mgsm_L2_q4_k_m
01:44 started l2thinker_ctrl_mgsm_L2_f16
01:45 started l2thinker_ctrl_mgsm_L2_q4_k_m
01:46 started l2thinker_trim_aya_L2_f16
01:47 started l2thinker_trim_aya_L2_q4_k_m
01:48 finished l2thinker_trim_mgsm_L2_f16 rc 0 | live 5 | queued 2
01:48 finished l2thinker_trim_mgsm_L2_q4_k_m rc 0 | live 4 | queued 2
01:48 started l2thinker_ctrl_aya_L2_f16
01:49 started l2thinker_ctrl_aya_L2_q4_k_m
02:55 finished l2thinker_ctrl_aya_L2_f16 rc 0 | live 5 | queued 0
02:59 finished l2thinker_ctrl_aya_L2_q4_k_m rc 0 | live 4 | queued 0
04:38 finished l2thinker_ctrl_mgsm_L2_f16 rc 0 | live 3 | queued 0
04:42 finished l2thinker_trim_aya_L2_f16 rc -15 | live 2 | queued 0
04:42 finished l2thinker_trim_aya_L2_q4_k_m rc -15 | live 1 | queued 0
04:46 finished l2thinker_ctrl_mgsm_L2_q4_k_m rc 0 | live 0 | queued 0


AssertionError: failed: ['l2thinker_trim_aya_L2_f16', 'l2thinker_trim_aya_L2_q4_k_m'] (see logs/gen_<name>.log)

In [29]:
!mkdir -p /content/drive/MyDrive/tinyaya-eval/runs_l2 && cp -rv /content/tinyaya-eval/runs_l2/l2thinker_trim /content/tinyaya-eval/runs_l2/l2thinker_ctrl /content/drive/MyDrive/tinyaya-eval/runs_l2/

'/content/tinyaya-eval/runs_l2/l2thinker_trim/trim/mgsm__L2__f16.jsonl' -> '/content/drive/MyDrive/tinyaya-eval/runs_l2/l2thinker_trim/trim/mgsm__L2__f16.jsonl'
'/content/tinyaya-eval/runs_l2/l2thinker_trim/trim/mgsm__L2__q4_k_m.jsonl' -> '/content/drive/MyDrive/tinyaya-eval/runs_l2/l2thinker_trim/trim/mgsm__L2__q4_k_m.jsonl'
'/content/tinyaya-eval/runs_l2/l2thinker_trim/trim_a100/aya__L2__f16.jsonl' -> '/content/drive/MyDrive/tinyaya-eval/runs_l2/l2thinker_trim/trim_a100/aya__L2__f16.jsonl'
'/content/tinyaya-eval/runs_l2/l2thinker_trim/trim_a100/aya__L2__q4_k_m.jsonl' -> '/content/drive/MyDrive/tinyaya-eval/runs_l2/l2thinker_trim/trim_a100/aya__L2__q4_k_m.jsonl'
'/content/tinyaya-eval/runs_l2/l2thinker_trim/trim_a100/mgsm__L2__f16.jsonl' -> '/content/drive/MyDrive/tinyaya-eval/runs_l2/l2thinker_trim/trim_a100/mgsm__L2__f16.jsonl'
'/content/tinyaya-eval/runs_l2/l2thinker_trim/trim_a100/mgsm__L2__q4_k_m.jsonl' -> '/content/drive/MyDrive/tinyaya-eval/runs_l2/l2thinker_trim/trim_a100/mgsm

## 16 · KV-cache optimisation (roadmap C, exploratory)

Every log shows `llama_kv_cache_iswa: using full-size SWA cache`: 3 of 4 layers use a 4096-token sliding window but the cache is allocated at full `n_ctx`. Two knobs, measured for memory and output identity on 20 Aya prompts: `swa_full=False` (llama.cpp PR #13194) and KV quantization `type_k/type_v = q8_0` (V needs flash attention). Memory at 4K and 32K contexts. **Identity must be 100% at f16 for swa_full and is not expected for q8_0 KV (report it).** This is the setting that matters most for a 32K reasoning model on a phone.

In [ ]:
import gc, llama_cpp
from llama_cpp import Llama
def vram(): return int(subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip().splitlines()[0])
prompts = [it["prompt"] for it in l2.load_items("aya", ["bn"], 20)]
STOP = ["<|END_OF_TURN_TOKEN|>", "<|END_RESPONSE|>"]
def run(cfg_, n_ctx):
    kw = {k: v for k, v in cfg_.items() if k != "label"}
    base = vram()
    M = Llama(model_path=str(GGUF / f"{BASE}-q4_k_m.gguf"), n_ctx=n_ctx, n_gpu_layers=-1, verbose=False, **kw)
    used = vram() - base
    outs = []
    for p in prompts:
        M.reset(); r = M.create_chat_completion([{"role": "user", "content": p}], max_tokens=512, temperature=0.0, stop=STOP)
        outs.append(r["choices"][0]["message"]["content"])
    del M; gc.collect()
    return used, outs
configs = [{"label": "default"}, {"label": "swa_full=False", "swa_full": False}, {"label": "kv q8_0 + flash", "type_k": llama_cpp.GGML_TYPE_Q8_0, "type_v": llama_cpp.GGML_TYPE_Q8_0, "flash_attn": True}]
res = {}
for n_ctx in (4096, 32768):
    ref = None
    for c in configs:
        try:
            used, outs = run(c, n_ctx)
        except Exception as e:
            print(f"n_ctx={n_ctx:5d} {c['label']:18s} FAILED: {str(e)[:120]}"); continue
        ref = ref or outs
        ident = 100 * sum(a == b for a, b in zip(ref, outs)) / len(outs)
        print(f"n_ctx={n_ctx:5d} {c['label']:18s} VRAM +{used/1024:5.2f} GB  identity vs default {ident:5.1f}%")
print("\nnote: KV memory at 32K ~ 2.4 GB/stream at f16 before SWA savings; the number to beat is the default row at 32768")

## 17 · Status and next

Fill in after the run: keep %, size/speed deltas, identity, and whether item F ships. If it passes, the trimmed q4_k_m GGUF + `_ctrl` rows are the deployment candidate; feed it into `l2thinker_quant.ipynb` Part 7 as a fourth precision to confirm the tier pattern is unchanged by trimming.

# Save session

In [ ]:
# save what the A100 session needs: base GGUFs (if not already on Drive) + the trim selection files
import shutil
from pathlib import Path
D, L = Path("/content/drive/MyDrive/tinyaya-eval"), Path("/content/tinyaya-eval")
(D / "models" / "gguf").mkdir(parents=True, exist_ok=True)
for p in sorted((L / "models" / "gguf").glob("l2thinker-*.gguf")):          # base model only; trimmed ones are rebuilt
    dst = D / "models" / "gguf" / p.name
    if dst.exists() and dst.stat().st_size == p.stat().st_size: print("already on Drive:", p.name); continue
    print("copying", p.name); shutil.copy2(p, dst)
shutil.copytree(L / "calib", D / "calib", dirs_exist_ok=True); print("saved calib/ (keep file)")
for sub in ("runs_l2", "results", "logs"):
    if (L / sub).exists(): shutil.copytree(L / sub, D / sub, dirs_exist_ok=True)
print(sorted(p.name for p in (D / "models" / "gguf").glob("l2thinker*")))
print(sorted(str(p.relative_to(D)) for p in (D / "calib").rglob("*keep*.json")))

already on Drive: l2thinker-f16.gguf
already on Drive: l2thinker-q4_0.gguf
already on Drive: l2thinker-q4_k_m.gguf
already on Drive: l2thinker-q8_0.gguf
saved calib/ (keep file)
['l2thinker-f16.gguf', 'l2thinker-q4_0.gguf', 'l2thinker-q4_k_m.gguf', 'l2thinker-q8_0.gguf', 'l2thinker_trim-f16.gguf.trim.json']
['calib/trim/fire/fire_keep.json', 'calib/trim/fire/fire_trim2_keep.json', 'calib/trim/l2thinker/l2thinker_trim_keep.json']
